# Task 2–3 — Sentence-BERT (Siamese) for NLI + Evaluation
**Goal:** load the MLM-trained BERT encoder from Task 1, wrap it in a siamese architecture to produce sentence embeddings, and fine-tune on **SNLI/MNLI** with the **Softmax classification objective**:

\( o = softmax(W^T [u, v, |u-v|]) \)

Artifacts produced:
- `artifacts/sbert_nli_model.pt` (classifier + encoder weights)
- `artifacts/sbert_nli_tokenizer/` (tokenizer)


In [16]:
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, BertTokenizerFast
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import classification_report
from transformers import get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Correct paths to save the model and tokenizer
model_save_path = os.path.join('..', 'artifacts', 'sbert_nli_model.pt')
tokenizer_save_path = os.path.join('..', 'artifacts', 'sbert_nli_tokenizer')

# Ensure the directories exist
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
os.makedirs(tokenizer_save_path, exist_ok=True)


device: cuda


## 1) Load SNLI (or MNLI)

In [17]:
raw = load_dataset('snli')
raw = raw.filter(lambda x: x['label'] != -1)  # remove samples with label = -1 (no gold label)
raw

label2name = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
name2label = {v: k for k, v in label2name.items()}

## 2) Tokenizer + preprocessing
We encode **premise** and **hypothesis** separately (two towers).

In [18]:
MODEL_DIR = '../artifacts/bert_mlm_model'  # output from Task 1 (outside notebooks folder)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

MAX_LEN = 128
def preprocess(batch):
    p = tokenizer(batch['premise'], truncation=True, padding='max_length', max_length=MAX_LEN)
    h = tokenizer(batch['hypothesis'], truncation=True, padding='max_length', max_length=MAX_LEN)
    out = {
        'p_input_ids': p['input_ids'],
        'p_attention_mask': p['attention_mask'],
        'h_input_ids': h['input_ids'],
        'h_attention_mask': h['attention_mask'],
        'labels': batch['label']
    }
    return out

tok = raw.map(preprocess, batched=True, remove_columns=raw['train'].column_names)
tok = tok.rename_column('labels', 'label')
tok.set_format(type='torch')
tok

Map: 100%|██████████| 9842/9842 [00:01<00:00, 8880.29 examples/s] 


DatasetDict({
    test: Dataset({
        features: ['p_input_ids', 'p_attention_mask', 'h_input_ids', 'h_attention_mask', 'label'],
        num_rows: 9824
    })
    validation: Dataset({
        features: ['p_input_ids', 'p_attention_mask', 'h_input_ids', 'h_attention_mask', 'label'],
        num_rows: 9842
    })
    train: Dataset({
        features: ['p_input_ids', 'p_attention_mask', 'h_input_ids', 'h_attention_mask', 'label'],
        num_rows: 549367
    })
})

## 3) Model: Siamese encoder + mean pooling + SoftmaxLoss head

In [19]:
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

class SBERT_NLI(nn.Module):
    def __init__(self, encoder_name_or_path, num_labels=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name_or_path)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden * 3, num_labels)

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        u = mean_pool(out.last_hidden_state, attention_mask)
        return u

    def forward(self, p_input_ids, p_attention_mask, h_input_ids, h_attention_mask, labels=None):
        u = self.encode(p_input_ids, p_attention_mask)
        v = self.encode(h_input_ids, h_attention_mask)
        uv_abs = torch.abs(u - v)
        feats = torch.cat([u, v, uv_abs], dim=1)
        logits = self.classifier(feats)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {'loss': loss, 'logits': logits, 'u': u, 'v': v}

model = SBERT_NLI(MODEL_DIR).to(device)
sum(p.numel() for p in model.parameters())


Loading weights: 100%|██████████| 69/69 [00:00<00:00, 872.39it/s, Materializing param=encoder.layer.3.output.dense.weight]              
BertModel LOAD REPORT from: ../artifacts/bert_mlm_model
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


10940931

## 4) Dataloaders

In [20]:
batch_size = 32
train_dl = DataLoader(tok['train'], batch_size=batch_size, shuffle=True)
val_dl = DataLoader(tok['validation'], batch_size=batch_size)
test_dl = DataLoader(tok['test'], batch_size=batch_size)


## 5) Train

In [21]:
lr = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
num_epochs = 2
total_steps = num_epochs * len(train_dl)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

from tqdm.auto import tqdm
for epoch in range(num_epochs):
    model.train()
    pbar = tqdm(train_dl, desc=f'train epoch {epoch + 1}')
    total_loss = 0
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(
            p_input_ids=batch['p_input_ids'],
            p_attention_mask=batch['p_attention_mask'],
            h_input_ids=batch['h_input_ids'],
            h_attention_mask=batch['h_attention_mask'],
            labels=batch['label']
        )
        loss = out['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        pbar.set_postfix(loss=total_loss / (pbar.n + 1))

    # quick validation accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(
                p_input_ids=batch['p_input_ids'],
                p_attention_mask=batch['p_attention_mask'],
                h_input_ids=batch['h_input_ids'],
                h_attention_mask=batch['h_attention_mask']
            )['logits']
            pred = logits.argmax(dim=1)
            correct += (pred == batch['label']).sum().item()
            total += pred.numel()
    print('val acc:', correct / total)

train epoch 1: 100%|██████████| 17168/17168 [20:17:18<00:00,  4.25s/it, loss=0.85]        


val acc: 0.6960983539930908


train epoch 2: 100%|██████████| 17168/17168 [25:15<00:00, 11.33it/s, loss=0.712]


val acc: 0.7190611664295875


## 6) Task 3 — Evaluation (Classification Report)

In [22]:
def predict(dataloader):
    model.eval()
    all_pred = []
    all_gold = []
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(
                p_input_ids=batch['p_input_ids'],
                p_attention_mask=batch['p_attention_mask'],
                h_input_ids=batch['h_input_ids'],
                h_attention_mask=batch['h_attention_mask']
            )['logits']
            all_pred.extend(logits.argmax(dim=1).cpu().tolist())
            all_gold.extend(batch['label'].cpu().tolist())
    return all_gold, all_pred

gold, pred = predict(test_dl)
print(classification_report(gold, pred, target_names=[label2name[i] for i in range(3)]))

               precision    recall  f1-score   support

   entailment       0.72      0.79      0.75      3368
      neutral       0.70      0.66      0.68      3219
contradiction       0.74      0.70      0.72      3237

     accuracy                           0.72      9824
    macro avg       0.72      0.72      0.72      9824
 weighted avg       0.72      0.72      0.72      9824



## 7) Save for Task 4 (Web app)

In [23]:
torch.save({'model_state_dict': model.state_dict(), 'encoder_dir': MODEL_DIR}, model_save_path)
tokenizer.save_pretrained(tokenizer_save_path)
print(f"Saved model to {model_save_path}")
print(f"Saved tokenizer to {tokenizer_save_path}")

Saved model to ..\artifacts\sbert_nli_model.pt
Saved tokenizer to ..\artifacts\sbert_nli_tokenizer


## 8) Inference helper (cosine similarity + NLI)

In [24]:
import torch.nn.functional as F

def nli_predict(premise, hypothesis):
    model.eval()
    p = tokenizer(premise, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt').to(device)
    h = tokenizer(hypothesis, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt').to(device)
    out = model(p['input_ids'], p['attention_mask'], h['input_ids'], h['attention_mask'])
    probs = F.softmax(out['logits'], dim=1)[0]
    label = probs.argmax().item()
    # cosine similarity of pooled embeddings
    u = out['u'][0].detach().cpu().numpy().reshape(1, -1)
    v = out['v'][0].detach().cpu().numpy().reshape(1, -1)
    cos = float((u @ v.T) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9))
    return {'label': label2name[label], 'probs': probs.detach().cpu().tolist(), 'cosine': cos}